# Model C: Time-Series + Transformer (8 encoder)

Model C memakai baris dan split yang sama dengan Model A dan B. Teks berita diambil
dari `data/export/articles.parquet`, lalu embedding judul diagregasi per
`aligned_trading_date` (hari bursa versi WIB) dengan mean pooling dan L2 normalization.
Hari tanpa berita diisi vektor nol. Semua artefak ditulis ke `runs/model_c_v1/`.

| Encoder | Repo | Pooling | Dim |
|---|---|---|---|
| `finbert` | ProsusAI/finbert | cls | 768 |
| `roberta` | distilroberta-base | cls | 768 |
| `bge_base` | BAAI/bge-base-en-v1.5 | cls | 768 |
| `harrier270m` | microsoft/harrier-oss-v1-270m | last | 640 |
| `qwen3_06b` | Qwen/Qwen3-Embedding-0.6B | last | 1024 |
| `granite_r2` | ibm-granite/granite-embedding-english-r2 | cls | 768 |
| `jina_v5_nano` | jinaai/jina-embeddings-v5-text-nano-classification | last | 768 |
| `finmodernbert` | BatuhanECB/FinModernBERT-embed-large | mean | 1024 |

| Varian | Fusion | Fitur |
|---|---|---|
| `A` | - | 7 fitur kurs |
| `C_<encoder>_early` | Early | 7 kurs + 64 komponen PCA encoder |
| `C_combined` | Early | 7 kurs + 64 FinBERT + 64 DistilRoBERTa |
| `C_all` | Early | 7 kurs + 64 komponen PCA tiap encoder |
| `C_<encoder>_late`, `C_combined_late`, `C_all_late` | Late | alpha x prediksi kurs + (1-alpha) x prediksi teks, alpha dari Val |

PCA dan scaler hanya di-fit di train. Encoder dimuat satu per satu dalam fp16,
batch diturunkan otomatis bila OOM.


## 0. Import dan lokasi file

In [ ]:
import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModel

import sklearn, xgboost, lightgbm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    AdaBoostRegressor, GradientBoostingRegressor,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.manifold import TSNE
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED = 42
N_PCA = 64
BATCH_SIZE = 64
MAX_LENGTH = 64
USE_FP16 = True
# gemma3 (harrier) meluap di fp16: smoke test CP3 menghasilkan NaN/Inf.
FORCE_FP32 = {"harrier270m"}
# Mesin 20 core / 11,7 GB: n_jobs=-1 di GridSearchCV membunuh worker
# (BrokenProcessPool), jadi paralelisme dibatasi.
N_JOBS = 4
N_JOBS_INNER = 1

# tag: (repo, pooling). "cls" = token pertama, "last" = token terakhir non-pad,
# "mean" = rata-rata token non-pad. Mengikuti 1_Pooling/config.json tiap repo.
EMB_MODELS = {
    "finbert":       ("ProsusAI/finbert", "cls"),
    "roberta":       ("distilroberta-base", "cls"),
    "bge_base":      ("BAAI/bge-base-en-v1.5", "cls"),
    "harrier270m":   ("microsoft/harrier-oss-v1-270m", "last"),
    "qwen3_06b":     ("Qwen/Qwen3-Embedding-0.6B", "last"),
    "granite_r2":    ("ibm-granite/granite-embedding-english-r2", "cls"),
    "jina_v5_nano":  ("jinaai/jina-embeddings-v5-text-nano-classification", "last"),
    "finmodernbert": ("BatuhanECB/FinModernBERT-embed-large", "mean"),
}
ENCODERS_TO_RUN = list(EMB_MODELS)

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.6g}")


def find_root(start):
    # Notebook bisa dijalankan dari folder mana pun, jadi root dicari ke atas.
    for cand in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (cand / "data" / "export" / "prices_daily.parquet").exists():
            return cand
    raise FileNotFoundError("Folder proyek (berisi data/export/prices_daily.parquet) tidak ditemukan")


ROOT = find_root(Path.cwd())
EXPORT_DIR = ROOT / "data" / "export"
LEGACY_DIR = ROOT / "trial-and-error" / "Ken" / "results"
RUN_DIR = ROOT / "trial-and-error" / "Ken" / "runs" / "model_c_v1"
OUT_DIR = RUN_DIR / "outputs_c"
CACHE_DIR = RUN_DIR / "cache"
LOG_DIR = RUN_DIR / "logs"
for d in (OUT_DIR, CACHE_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads()))

sys.path.insert(0, str((ROOT / "src").resolve()))
from eval_utils import evaluate, METRICS

print("Proyek   :", ROOT)
print("RUN_DIR  :", RUN_DIR)
print("OUT_DIR  :", OUT_DIR)
print("Metrik   :", METRICS)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| device", DEVICE)
if DEVICE == "cuda":
    print("GPU", torch.cuda.get_device_name(0), "| VRAM",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
print("Encoder  :", ENCODERS_TO_RUN)
print("Batch", BATCH_SIZE, "| max_len", MAX_LENGTH, "| fp16", bool(USE_FP16 and DEVICE == "cuda"))


## 1. Kurs dan split

In [ ]:
prices = pd.read_parquet(EXPORT_DIR / "prices_daily.parquet")
prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
prices = prices.sort_values("date").reset_index(drop=True)

train = prices[prices["split"] == "train"].reset_index(drop=True)
val = prices[prices["split"] == "val"].reset_index(drop=True)
test = prices[prices["split"] == "test"].reset_index(drop=True)

for name, part in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"{name:5}: {len(part):4d} | {part['date'].min().date()} -> {part['date'].max().date()}")

FEAT_A = ["kurs_lag1", "kurs_lag2", "kurs_lag3", "kurs_lag4", "kurs_lag5",
          "kurs_roll_mean5", "kurs_roll_std5"]
ytr, yva, yte = train["target"].values, val["target"].values, test["target"].values

# dataset_model_A.parquet tetap dibaca dari folder hasil lama (read-only).
a = pd.read_parquet(LEGACY_DIR / "dataset_model_A.parquet")
a["date"] = pd.to_datetime(a["Tanggal_dt"]).dt.normalize()
cea = prices[["date", "target"]].merge(a[["date", "target"]], on="date", suffixes=("", "_A"))
print("tanggal cocok dengan Model A:", len(cea), "dari", len(prices))
print("selisih target maks:", float((cea["target"] - cea["target_A"]).abs().max()))


## 2. Artikel dan embedding harian

In [ ]:
articles = pd.read_parquet(EXPORT_DIR / "articles.parquet")
articles["tanggal_bursa"] = pd.to_datetime(articles["aligned_trading_date"]).dt.normalize()
articles = articles.sort_values(["tanggal_bursa", "published_wib"]).reset_index(drop=True)

in_calendar = articles["tanggal_bursa"].isin(prices["date"])
print("artikel total          :", len(articles))
print("di luar kalender bursa :", int((~in_calendar).sum()))
print("hari bursa berita      :", articles.loc[in_calendar, "tanggal_bursa"].nunique())
print("berita per hari        : median", float(articles.groupby("tanggal_bursa").size().median()))

DATES_UNIQUE = pd.unique(articles["tanggal_bursa"])
GROUPS = [articles["tanggal_bursa"].values == d for d in DATES_UNIQUE]


def pool_hidden(h, mask, mode):
    if mode == "cls":
        return h[:, 0]
    if mode == "mean":
        m = mask.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)
    if mode == "last":
        idx = (mask.sum(1) - 1).clamp(min=0)
        return h[torch.arange(h.size(0), device=h.device), idx]
    raise ValueError(f"pooling tidak dikenal: {mode}")


def load_encoder(model_name, dtype):
    try:
        model = AutoModel.from_pretrained(model_name, dtype=dtype)
    except TypeError:
        model = AutoModel.from_pretrained(model_name, torch_dtype=dtype)
    return model.to(DEVICE).eval()


def encode_titles(texts, model_name, pooling, dtype, batch_size=BATCH_SIZE):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "right"  # "last" memakai indeks mask.sum(1)-1
    model = load_encoder(model_name, dtype)
    chunks, start, oom, t0 = [], 0, 0, time.time()
    while start < len(texts):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH,
                        return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        try:
            with torch.no_grad():
                out = model(**enc).last_hidden_state
            vec = pool_hidden(out.float(), enc["attention_mask"], pooling)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            oom += 1
            batch_size = max(4, batch_size // 2)
            print(f"  OOM {model_name}: batch -> {batch_size} (ke-{oom})")
            if oom >= 3:
                raise
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            continue
        chunks.append(torch.nn.functional.normalize(vec, dim=1).cpu().numpy())
        start += len(batch)
        if start % 1024 < batch_size:
            print(f"   {min(start, len(texts))}/{len(texts)}")
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"  {model_name}: {len(texts)} teks dalam {time.time() - t0:.1f}s")
    return np.vstack(chunks).astype(np.float32)


def daily_embeddings(tag, model_name, pooling):
    cache = CACHE_DIR / f"embeddings_daily_{tag}.parquet"
    meta_path = CACHE_DIR / f"embeddings_daily_{tag}.json"
    if cache.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if (meta.get("model") == model_name and meta.get("n_artikel") == len(articles)
                and meta.get("pooling") == pooling):
            print(f"{tag:14}: memakai cache {cache.name}")
            return pd.read_parquet(cache)

    texts = articles["title"].str.strip().tolist()
    half = DEVICE == "cuda" and USE_FP16 and tag not in FORCE_FP32
    dtype = torch.float16 if half else torch.float32
    try:
        vec = encode_titles(texts, model_name, pooling, dtype)
        if not np.isfinite(vec).all():
            raise FloatingPointError("embedding mengandung NaN/Inf")
    except Exception as e:
        print(f"{tag:14}: gagal ({type(e).__name__}: {str(e)[:90]}), ulang fp32")
        vec = encode_titles(texts, model_name, pooling, torch.float32)

    rows = [vec[m].mean(axis=0) for m in GROUPS]
    daily = pd.DataFrame(np.vstack(rows), index=DATES_UNIQUE)
    daily = daily.div(np.linalg.norm(daily.values, axis=1, keepdims=True), axis=0)
    daily.index.name = "date"
    daily.to_parquet(cache)
    meta_path.write_text(json.dumps({"model": model_name, "n_artikel": int(len(articles)),
                                     "dim": int(daily.shape[1]), "pooling": pooling},
                                    indent=2), encoding="utf-8")
    print(f"{tag:14}: tersimpan {cache.name} {daily.shape}")
    return daily


DAILY, EMB_FAILED = {}, {}
for tag in ENCODERS_TO_RUN:
    repo, pooling = EMB_MODELS[tag]
    try:
        DAILY[tag] = daily_embeddings(tag, repo, pooling)
    except Exception as e:
        EMB_FAILED[tag] = f"{type(e).__name__}: {str(e)[:200]}"
        print(f"{tag:14}: GAGAL {EMB_FAILED[tag][:120]}")

print("encoder aktif:", list(DAILY))
if EMB_FAILED:
    print("encoder gagal:", EMB_FAILED)
if not DAILY:
    raise RuntimeError("Tidak ada encoder yang berhasil")
for tag, e in DAILY.items():
    print(f"{tag:14}: {e.shape[0]} hari, dim {e.shape[1]}")


## 3. Early fusion: kurs + PCA embedding

In [ ]:
def embedding_matrix(part, tag):
    M = DAILY[tag].reindex(part["date"]).values.astype(float)
    miss = np.isnan(M).any(axis=1)
    if miss.any():
        print(f"  {tag}: {int(miss.sum())} hari tanpa berita diisi 0")
    return np.nan_to_num(M, nan=0.0)


emb_scaler = {tag: StandardScaler().fit(embedding_matrix(train, tag)) for tag in DAILY}
RAW = {(tag, s): emb_scaler[tag].transform(embedding_matrix(p, tag))
       for tag in DAILY for s, p in [("train", train), ("val", val), ("test", test)]}

PCA = {}
for tag in DAILY:
    svd = TruncatedSVD(n_components=N_PCA, random_state=SEED)
    PCA[(tag, "train")] = svd.fit_transform(RAW[(tag, "train")])
    for s in ("val", "test"):
        PCA[(tag, s)] = svd.transform(RAW[(tag, s)])
    print(f"{tag:14}: PCA {N_PCA} komponen, explained={svd.explained_variance_ratio_.sum():.3f}")

ts_scaler = StandardScaler().fit(train[FEAT_A])
TS = {s: ts_scaler.transform(p[FEAT_A]) for s, p in [("train", train), ("val", val), ("test", test)]}

# Varian early fusion: satu per encoder, plus gabungan finbert+roberta (nama lama
# C_combined supaya tetap sebanding dengan laporan sebelumnya) dan C_all.
COMPOSITION = {f"C_{tag}_early": [tag] for tag in DAILY}
if {"finbert", "roberta"} <= set(DAILY):
    COMPOSITION["C_combined"] = ["finbert", "roberta"]
COMPOSITION["C_all"] = list(DAILY)

EF = {}
for var, tags in COMPOSITION.items():
    for s in ("train", "val", "test"):
        EF[(var, s)] = np.hstack([TS[s]] + [PCA[(tag, s)] for tag in tags])

for var in COMPOSITION:
    print(f"{var:22}: {EF[(var, 'train')].shape[1]} fitur ({len(COMPOSITION[var])} encoder)")


## 4. Model, evaluasi, dan pelatihan

In [ ]:
MODELS = {
    "LinearRegression": lambda: LinearRegression(),
    "Ridge":            lambda: Ridge(),
    "Lasso":            lambda: Lasso(),
    "ElasticNet":       lambda: ElasticNet(),
    "BayesianRidge":    lambda: BayesianRidge(),
    "KNN":              lambda: KNeighborsRegressor(),
    "SVR":              lambda: SVR(),
    "DecisionTree":     lambda: DecisionTreeRegressor(random_state=SEED),
    "RandomForest":     lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "ExtraTrees":       lambda: ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    "AdaBoost":         lambda: AdaBoostRegressor(random_state=SEED),
    "GradientBoosting": lambda: GradientBoostingRegressor(random_state=SEED),
    "XGBoost":          lambda: XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
    "LightGBM":         lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1),
}

VARIANTS = {"A": TS}
for var in COMPOSITION:
    VARIANTS[var] = {s: EF[(var, s)] for s in ("train", "val", "test")}

rows, preds = [], []


def add_row(varian, model_name, split, part, y, y_pred, n_fitur):
    rows.append({"Varian": varian, "Model": model_name, "Split": split,
                 "n_fitur": n_fitur, **evaluate(y, y_pred)})
    preds.append(pd.DataFrame({"Varian": varian, "Model": model_name, "Split": split,
                               "date": part["date"].values, "y_true": y, "y_pred": y_pred}))


for varian, X in VARIANTS.items():
    t0 = time.time()
    n_fitur = X["train"].shape[1]
    for model_name, make_model in MODELS.items():
        model = make_model().fit(X["train"], ytr)
        for split, part, y in [("Val", val, yva), ("Test", test, yte)]:
            add_row(varian, model_name, split, part, y, model.predict(X[split.lower()]), n_fitur)
    print(f"selesai: {varian:22} | {n_fitur:4d} fitur | {time.time() - t0:.1f}s")

results = pd.DataFrame(rows)
results["Degenerate"] = results["PredStd"] < 1e-9
print("total baris:", len(results), "| degenerate:", int(results["Degenerate"].sum()))


## 5. Late fusion

In [ ]:
ts_model = BayesianRidge().fit(TS["train"], ytr)
pred_ts = {s: ts_model.predict(TS[s]) for s in ("train", "val", "test")}

LATE_TAGS = {f"C_{tag}_late": [tag] for tag in DAILY}
if {"finbert", "roberta"} <= set(DAILY):
    LATE_TAGS["C_combined_late"] = ["finbert", "roberta"]
LATE_TAGS["C_all_late"] = list(DAILY)

LATE = {}
for var, tags in LATE_TAGS.items():
    Xe = {s: np.hstack([RAW[(t, s)] for t in tags]) for s in ("train", "val", "test")}
    text_model = Ridge(alpha=100.0).fit(Xe["train"], ytr)
    pred_txt = {s: text_model.predict(Xe[s]) for s in ("train", "val", "test")}

    best_alpha, best_rmse = 0.5, np.inf
    for alpha in np.linspace(0, 1, 101):
        rmse = np.sqrt(np.mean((alpha * pred_ts["val"] + (1 - alpha) * pred_txt["val"] - yva) ** 2))
        if rmse < best_rmse:
            best_rmse, best_alpha = rmse, alpha

    n_fitur = len(FEAT_A) + Xe["train"].shape[1]
    LATE[var] = {"alpha": best_alpha, "pred": pred_txt, "n_fitur": n_fitur, "tags": tags}
    for split, part, y, key in [("Val", val, yva, "val"), ("Test", test, yte, "test")]:
        yp = best_alpha * pred_ts[key] + (1 - best_alpha) * pred_txt[key]
        add_row(var, f"Ridge_alpha{best_alpha:.2f}", split, part, y, yp, n_fitur)
    print(f"{var:22}: alpha_ts={best_alpha:.2f}, Val RMSE={best_rmse:.6f}, "
          f"dim teks={Xe['train'].shape[1]}")

results = pd.DataFrame(rows)
results["Degenerate"] = results["PredStd"] < 1e-9
print("total baris setelah late fusion:", len(results))


## 6. Hasil per varian

In [7]:
for split in ("Val", "Test"):
    t = (results[(results["Split"] == split) & (~results["Degenerate"])]
         .sort_values(["Varian", "RMSE"])
         .groupby("Varian").head(3)
         .set_index(["Varian", "Model"])[["n_fitur", "RMSE", "MAE", "DirAcc", "PredStd"]])
    print(f"\n=== {split} | 3 model terbaik per varian ===")
    print(t.to_string())


=== Val | 3 model terbaik per varian ===
                                  n_fitur       RMSE        MAE   DirAcc     PredStd
Varian          Model                                                               
A               BayesianRidge           7 0.00338869 0.00250102 0.448864  0.00025135
                Ridge                   7 0.00339263 0.00251092 0.460227 0.000353976
                LinearRegression        7 0.00340153 0.00252672 0.465909 0.000473432
C_combined      RandomForest          135 0.00346523 0.00258257 0.477273 0.000716911
                GradientBoosting      135 0.00347231 0.00265876 0.477273  0.00106372
                ExtraTrees            135   0.003474 0.00258458 0.556818 0.000878208
C_combined_late Ridge_alpha0.97      1543 0.00338795 0.00250512 0.448864 0.000255795
C_finbert_early BayesianRidge          71 0.00346571 0.00263871 0.465909 0.000866808
                Ridge                  71 0.00348544 0.00270091 0.471591  0.00100057
                LinearR

## 7. Delta RMSE terhadap Model A

In [ ]:
base = (results[(results["Varian"] == "A") & (results["Split"] == "Test")]
        .set_index("Model")["RMSE"])
best_a = results[(results["Varian"] == "A") & (results["Split"] == "Test")
                 & (~results["Degenerate"])].sort_values("RMSE").iloc[0]
print(f"Anchor A (Test): {best_a['Model']} RMSE={best_a['RMSE']:.7f} DirAcc={best_a['DirAcc']:.4f}")

delta_rows = []
for var in sorted(v for v in results["Varian"].unique() if v.startswith("C_")):
    sub = results[(results["Varian"] == var) & (results["Split"] == "Test") & (~results["Degenerate"])]
    if sub.empty:
        continue
    best = sub.sort_values("RMSE").iloc[0]
    delta_rows.append({"Varian": var, "Model": best["Model"], "n_fitur": int(best["n_fitur"]),
                       "RMSE": best["RMSE"], "DirAcc": best["DirAcc"],
                       "dRMSE_pct": 100 * (best["RMSE"] - best_a["RMSE"]) / best_a["RMSE"]})
delta = pd.DataFrame(delta_rows).sort_values("dRMSE_pct")
print("\n=== Delta C vs A (Test), urut dari terbaik ===")
print(delta.to_string(index=False, float_format=lambda x: f"{x:.6g}"))
delta.to_csv(OUT_DIR / "delta_vs_A_model_C.csv", index=False)


## 8. Hyperparameter tuning

In [ ]:
GRIDS = {
    "Ridge": {"alpha": [0.1, 1.0, 10.0, 100.0]},
    "Lasso": {"alpha": [1e-4, 1e-3, 1e-2]},
    "ElasticNet": {"alpha": [1e-4, 1e-3], "l1_ratio": [0.2, 0.5, 0.8]},
    "KNN": {"n_neighbors": [3, 5, 7, 11, 21], "weights": ["uniform", "distance"]},
    "SVR": {"C": [0.1, 1.0, 10.0], "epsilon": [1e-4, 1e-3]},
    "RandomForest": {"max_depth": [3, 5, None], "min_samples_leaf": [1, 5]},
    "XGBoost": {"max_depth": [2, 3], "learning_rate": [0.03, 0.05], "n_estimators": [300]},
    "LightGBM": {"num_leaves": [7, 15], "learning_rate": [0.03, 0.05], "n_estimators": [300]},
}
CV = TimeSeriesSplit(n_splits=5)

tuned = {}
tune_rows = []
for var in COMPOSITION:
    t0 = time.time()
    sub = results[(results["Varian"] == var) & (results["Split"] == "Val") & (~results["Degenerate"])]
    model_name = sub.sort_values("RMSE").iloc[0]["Model"]
    if model_name not in GRIDS:
        model_name = "Ridge"

    X = VARIANTS[var]

    def make_for_tuning(name):
        est = MODELS[name]()
        try:
            est.set_params(n_jobs=N_JOBS_INNER)
        except ValueError:
            pass
        return est

    gs = GridSearchCV(make_for_tuning(model_name), GRIDS[model_name], cv=CV, n_jobs=N_JOBS,
                      scoring="neg_root_mean_squared_error")
    gs.fit(X["train"], ytr)
    best = gs.best_estimator_
    tuned[var] = {"model": best, "X": X}

    for split, part, y in [("Val", val, yva), ("Test", test, yte)]:
        yp = best.predict(X[split.lower()])
        tune_rows.append({"Varian": var + "_tuned", "Model": model_name + "+tuned",
                          "Split": split, "n_fitur": X["train"].shape[1],
                          "params": json.dumps(gs.best_params_),
                          "cv_RMSE": float(-gs.best_score_), **evaluate(y, yp)})
        add_row(var + "_tuned", model_name + "+tuned", split, part, y, yp, X["train"].shape[1])
    print(f"{var:22}: {model_name} {gs.best_params_} | cv RMSE {-gs.best_score_:.6f} "
          f"| {time.time() - t0:.1f}s")

tuning = pd.DataFrame(tune_rows)
tuning["Degenerate"] = tuning["PredStd"] < 1e-9
print(tuning[tuning["Split"] == "Test"][["Varian", "Model", "params", "cv_RMSE", "RMSE", "DirAcc"]]
      .to_string(index=False, float_format=lambda x: f"{x:.6g}"))


## 9. Permutation importance

In [ ]:
def feature_names(var):
    tags = COMPOSITION.get(var)
    if tags is None:
        return list(FEAT_A)
    names = list(FEAT_A)
    for tag in tags:
        names += [f"{tag}_pc{i}" for i in range(1, N_PCA + 1)]
    return names


imp_rows = []
for var, item in tuned.items():
    Xva = item["X"]["val"]
    r = permutation_importance(item["model"], Xva, yva, n_repeats=20, random_state=SEED,
                               scoring="neg_root_mean_squared_error")
    for feat, mean, std in sorted(zip(feature_names(var), r.importances_mean, r.importances_std),
                                  key=lambda t: -t[1]):
        imp_rows.append({"Varian": var, "Fitur": feat, "importance": mean, "std": std})

imp = pd.DataFrame(imp_rows)
for var in tuned:
    print(f"\n=== {var}: 8 fitur paling berpengaruh (validasi) ===")
    print(imp[imp["Varian"] == var].head(8).to_string(index=False, float_format=lambda x: f"{x:.6g}"))


## 10. t-SNE embedding harian dan volatilitas

In [ ]:
vol = prices[["date", "ret"]].copy()
vol["abs_ret"] = vol["ret"].abs()
q_hi = vol["abs_ret"].quantile(0.85)
q_lo = vol["abs_ret"].quantile(0.15)

TSNE_TAG = "finbert" if "finbert" in DAILY else next(iter(DAILY))
emb_dates = DAILY[TSNE_TAG].index.intersection(vol["date"])
X_emb = DAILY[TSNE_TAG].loc[emb_dates].values
vol_map = vol.set_index("date")["abs_ret"].reindex(emb_dates)
label = np.where(vol_map >= q_hi, "High Vol", np.where(vol_map <= q_lo, "Low Vol", None))
mask = label != None

X_plot = X_emb[mask]
labels = label[mask]
print("High Vol:", int((labels == "High Vol").sum()), "| Low Vol:", int((labels == "Low Vol").sum()))

if len(X_plot) >= 20:
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(30, len(X_plot) // 3),
                max_iter=1000, init="pca", learning_rate="auto")
    X_2d = tsne.fit_transform(X_plot)
    fig, ax = plt.subplots(figsize=(8, 6))
    for grp, color, marker in [("High Vol", "#b3231f", "o"), ("Low Vol", "#1f4e79", "s")]:
        m = labels == grp
        ax.scatter(X_2d[m, 0], X_2d[m, 1], c=color, marker=marker, s=28, alpha=0.75, label=grp)
    ax.set_title(f"t-SNE embedding judul {TSNE_TAG}\nVolatilitas harian USD/IDR: 15% teratas vs 15% terbawah")
    ax.set_xlabel("dim 1")
    ax.set_ylabel("dim 2")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"tsne_{TSNE_TAG}.png", dpi=120, bbox_inches="tight")
    print("Disimpan:", OUT_DIR / f"tsne_{TSNE_TAG}.png")


## 11. Analisis error per tanggal

In [12]:
best_c = results[(results["Varian"].str.startswith("C_")) & (results["Split"] == "Test")
                 & (~results["Degenerate"])].sort_values("RMSE").iloc[0]
pred_table = pd.concat(preds, ignore_index=True)

pa = pred_table[(pred_table["Varian"] == "A") & (pred_table["Model"] == best_a["Model"])
                & (pred_table["Split"] == "Test")].set_index("date")["y_pred"]
pc = pred_table[(pred_table["Varian"] == best_c["Varian"]) & (pred_table["Model"] == best_c["Model"])
                & (pred_table["Split"] == "Test")].set_index("date")["y_pred"]

err = pd.DataFrame({"y_true": yte, "pred_A": pa.values, "pred_C": pc.values}, index=test["date"].values)
err["err_A"] = (err["pred_A"] - err["y_true"]).abs()
err["err_C"] = (err["pred_C"] - err["y_true"]).abs()
nonzero = err["y_true"] != 0
improved = nonzero & (np.sign(err["pred_A"]) != np.sign(err["y_true"])) & (np.sign(err["pred_C"]) == np.sign(err["y_true"]))
worsened = nonzero & (np.sign(err["pred_A"]) == np.sign(err["y_true"])) & (np.sign(err["pred_C"]) != np.sign(err["y_true"]))
err["arah_membaik"] = improved
err["arah_memburuk"] = worsened

print(f"C terbaik (Test): {best_c['Varian']} / {best_c['Model']} RMSE={best_c['RMSE']:.7f}")
print("arah membaik:", int(improved.sum()), "hari | arah memburuk:", int(worsened.sum()), "hari",
      "| hari test:", int(nonzero.sum()))
err.to_csv(OUT_DIR / "error_analysis_model_C.csv")
print("\n5 hari dengan perbaikan arah terbesar:")
print(err[improved].nlargest(5, "err_A")[["y_true", "pred_A", "pred_C"]].to_string())

C terbaik (Test): C_combined / RandomForest RMSE=0.0028349
arah membaik: 47 hari | arah memburuk: 36 hari | hari test: 176

5 hari dengan perbaikan arah terbesar:
                y_true       pred_A       pred_C
2026-05-18   0.0096696 -0.000927105  0.000477749
2026-06-04  0.00600502 -0.000927105  0.000870038
2026-08-10 -0.00660919  6.10445e-05 -0.000379544
2026-03-10 -0.00561252  6.10445e-05 -0.000462479
2026-02-06  0.00361879  -0.00136867  0.000470456


## 12. Ekspor dan ringkasan

In [ ]:
results["params"] = ""
results["cv_RMSE"] = np.nan
all_metrics = pd.concat([results, tuning], ignore_index=True)
all_metrics.to_csv(OUT_DIR / "metrics_model_C.csv", index=False)

pred_df = pd.concat(preds, ignore_index=True)
pred_df[pred_df["Split"] == "Val"].to_csv(OUT_DIR / "predictions_val_model_C.csv", index=False)
pred_df[pred_df["Split"] == "Test"].to_csv(OUT_DIR / "predictions_test_model_C.csv", index=False)
tuning.to_csv(OUT_DIR / "tuning_model_C.csv", index=False)
imp.to_csv(OUT_DIR / "importance_model_C.csv", index=False)

meta = {
    "model": "C",
    "seed": SEED,
    "run_dir": str(RUN_DIR),
    "sumber": ["data/export/prices_daily.parquet", "data/export/articles.parquet"],
    "n_baris": int(len(prices)),
    "n_artikel": int(len(articles)),
    "split": {k: int((prices["split"] == k).sum()) for k in ("train", "val", "test")},
    "encoder": {tag: {"repo": repo, "pooling": pool, "dim": int(DAILY[tag].shape[1])}
                for tag, (repo, pool) in EMB_MODELS.items() if tag in DAILY},
    "encoder_gagal": EMB_FAILED,
    "embedding": {"teks": "judul", "agregasi": "mean harian + L2", "n_pca": N_PCA,
                  "alpha_late": {k: v["alpha"] for k, v in LATE.items()}},
    "versi": {"torch": torch.__version__, "transformers": __import__("transformers").__version__,
              "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
              "lightgbm": lightgbm.__version__},
}
(OUT_DIR / "metadata_model_C.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

summary = []
for var in ["A"] + list(COMPOSITION) + list(LATE_TAGS):
    sub = all_metrics[(all_metrics["Varian"] == var) & (all_metrics["Split"] == "Test")
                      & (~all_metrics["Degenerate"])]
    if sub.empty:
        continue
    best = sub.sort_values("RMSE").iloc[0]
    summary.append({"Varian": var, "Model": best["Model"], "n_fitur": int(best["n_fitur"]),
                    "RMSE": best["RMSE"], "MAE": best["MAE"], "DirAcc": best["DirAcc"]})
summary = pd.DataFrame(summary).sort_values("RMSE")
summary.to_csv(OUT_DIR / "summary_best_model_C.csv", index=False)
print("=== Model terbaik per varian (Test), urut RMSE ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.6g}"))
